# Phase 3b — Table 6 ablation (Kaggle)

## Before you run anything — two settings in the right-hand sidebar

Open **Settings** (or the `⋮` menu → *Accelerator* / *Internet*):

1. **Accelerator → GPU T4 x2** (or P100). Without this everything runs on CPU
   and would take about a day.
2. **Internet → On.** This is the one people miss. Without it `git clone` and
   `pip install` both fail with confusing network errors.

Both require a phone-verified Kaggle account — Settings → Phone Verification if
the toggles are greyed out.

Kaggle allows 30 GPU-hours per week and a 12-hour session, so this ~90 minute
run fits comfortably. Cell 1 checks both settings and stops with a clear message
if either is wrong.


## 1. Check the settings actually took effect


In [ ]:
import torch, socket, sys

gpu = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if gpu else 'NONE')

try:
    socket.create_connection(('huggingface.co', 443), timeout=8)
    net = True
except OSError:
    net = False
print('Internet:', 'on' if net else 'OFF')

if not gpu:
    sys.exit('Settings > Accelerator > GPU T4 x2, then re-run this cell.')
if not net:
    sys.exit('Settings > Internet > On, then re-run this cell.')
print('\nBoth settings correct — continue.')


## 2. Get the code


In [ ]:
%cd /kaggle/working
![ -d Echo ] && (cd Echo && git pull -q) || git clone -q https://github.com/ayn-aval/Echo.git
%cd /kaggle/working/Echo
!git log --oneline -1


## 3. Install

Kaggle ships torch already; only these are missing.


In [ ]:
!pip install -q -U transformers datasets scipy 2>&1 | tail -2
import transformers, datasets
print('transformers', transformers.__version__, '| datasets', datasets.__version__)


## 4. Run the ablation

Nine training runs at 100k pairs each — three pooling strategies and seven
concatenation variants, sharing one configuration. Roughly **90 minutes**.

`results/ablation.csv` is written after **every** run, so if the session dies
you lose one run rather than nine. Re-running this cell skips whatever is
already recorded.

Reduced scale is deliberate: nine full-size runs will not fit a free session,
and the ablation is about the *ordering* of configurations rather than their
absolute values. The subset size is printed with the results, not hidden.


In [ ]:
!python -m eval.ablation --model distilroberta-base --pairs 100000 \
                        --out-root /kaggle/working/ablation_models


## 5. Results

Copy the whole printed table below — including the two **CLAIM** lines, which
are the actual findings — back into the chat.


In [ ]:
import pandas as pd
df = pd.read_csv('results/ablation.csv')
df.to_csv('/kaggle/working/ablation.csv', index=False)   # saved as notebook output
print(df.to_string(index=False))


## If the session dies partway

`/kaggle/working` survives while the session lives, so simply re-running cell 4
continues. If the session is gone entirely, re-run cells 2–4 — you will lose the
completed runs and start over, so consider clicking **Save Version → Save & Run
All** instead, which runs it detached and keeps the output permanently.
